# J-Space Experiment — Phases 1–4 (Local Linux)

End-to-end launcher for a local Linux machine with CUDA. Each phase runs through the same CLI used from the terminal. Results are saved under one run root inside this repository.

Default configuration: **Qwen 3.5 9B smoke** (`configs/phase1_qwen35_9b_smoke.yaml`).

Recommended launch from the repository root:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -e '.[phase4,notebook]'
python -m ipykernel install --user --name jspace-research --display-name 'jspace-research'
jupyter notebook notebooks/JSpace_End_to_End_Local.ipynb
```

Then select the `jspace-research` kernel in Jupyter. The first code cell can also create `.venv/` and run `pip install` for you. Credentials can be entered manually in section 2 if they are not already in the shell environment.

## 1. Install dependencies and verify pinned checkouts

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

AGENTDOJO_REVISION = '089ed468cf3ed0322acc66b0211f26d9d90dbf60'
INJECAGENT_REVISION = 'f19c9f2c79a41046eb13c03c51a24c567a8ffa07'
PIP_EXTRAS = 'phase4,notebook'


def ensure_venv(repo_root: Path) -> Path:
    venv_dir = repo_root / '.venv'
    venv_python = (venv_dir / 'bin' / 'python').resolve()
    if not venv_python.is_file():
        subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)
    subprocess.run([str(venv_python), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
    subprocess.run(
        [str(venv_python), '-m', 'pip', 'install', '-e', f'{repo_root}[{PIP_EXTRAS}]'],
        check=True,
    )
    return venv_python


def verify_notebook_kernel(venv_python: Path) -> None:
    current_python = Path(sys.executable).resolve()
    if current_python != venv_python:
        raise RuntimeError(
            'This notebook is not running in the project virtualenv.\n'
            f'Expected: {venv_python}\n'
            f'Current:  {current_python}\n'
            'Activate the venv and select the jspace-research kernel, or launch with:\n'
            '  source .venv/bin/activate\n'
            '  jupyter notebook notebooks/JSpace_End_to_End_Local.ipynb'
        )


def verify_project_environment(venv_python: Path, venv_bin: Path) -> None:
    subprocess.run(
        [
            str(venv_python),
            '-c',
            (
                'import jspace_research, pandas, torch; '
                'from jspace_research.phase1.cli import main; '
                'print("environment ok")'
            ),
        ],
        check=True,
    )
    subprocess.run(
        [str(venv_bin / 'jspace-phase1'), '--help'],
        check=True,
        stdout=subprocess.DEVNULL,
    )


cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 'pyproject.toml').is_file() else cwd.parent
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Run this notebook from the repository root or notebooks/ directory.')

VENV_PYTHON = ensure_venv(REPO_ROOT)
VENV_BIN = REPO_ROOT / '.venv' / 'bin'
verify_notebook_kernel(VENV_PYTHON)
verify_project_environment(VENV_PYTHON, VENV_BIN)

BENCHMARKS_ROOT = Path(
    os.environ.get('JSPACE_BENCHMARKS_ROOT', REPO_ROOT.parent / 'jspace-benchmarks')
).expanduser().resolve()
BIPIA_CHECKOUT = REPO_ROOT / 'BIPIA'
AGENTDOJO_CHECKOUT = BENCHMARKS_ROOT / 'agentdojo'
INJECAGENT_CHECKOUT = BENCHMARKS_ROOT / 'InjecAgent'

subprocess.run(['git', 'submodule', 'update', '--init', 'BIPIA'], cwd=REPO_ROOT, check=True)

BENCHMARKS_ROOT.mkdir(parents=True, exist_ok=True)
if not AGENTDOJO_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/ethz-spylab/agentdojo.git', str(AGENTDOJO_CHECKOUT)],
        check=True,
    )
subprocess.run(['git', '-C', str(AGENTDOJO_CHECKOUT), 'checkout', AGENTDOJO_REVISION], check=True)
if not INJECAGENT_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/uiuc-kang-lab/InjecAgent.git', str(INJECAGENT_CHECKOUT)],
        check=True,
    )
subprocess.run(['git', '-C', str(INJECAGENT_CHECKOUT), 'checkout', INJECAGENT_REVISION], check=True)

print('Repository root:', REPO_ROOT)
print('Virtualenv python:', VENV_PYTHON)
print('Benchmarks root:', BENCHMARKS_ROOT)
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('AgentDojo revision:', subprocess.check_output(['git', '-C', str(AGENTDOJO_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())
print('InjecAgent revision:', subprocess.check_output(['git', '-C', str(INJECAGENT_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())

## 2. Enter credentials manually (optional)

Paste tokens here when they are not already available in the shell environment. Leave a field empty to keep using the existing environment variable instead.

In [ ]:
import os

# Paste tokens here if needed. Leave blank to use the shell environment instead.
MANUAL_HF_TOKEN = ''
MANUAL_OPENROUTER_API_KEY = ''


def apply_manual_credentials() -> None:
    if MANUAL_HF_TOKEN.strip():
        os.environ['HF_TOKEN'] = MANUAL_HF_TOKEN.strip()
    if MANUAL_OPENROUTER_API_KEY.strip():
        os.environ['OPENROUTER_API_KEY'] = MANUAL_OPENROUTER_API_KEY.strip()


apply_manual_credentials()

print('HF token set:', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))
print('OpenRouter key set:', bool(os.environ.get('OPENROUTER_API_KEY')))

## 3. Authenticate

In [ ]:
import os

from huggingface_hub import login

apply_manual_credentials()

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Hugging Face token loaded.')
elif MANUAL_HF_TOKEN.strip():
    raise RuntimeError('MANUAL_HF_TOKEN was set but HF_TOKEN is still missing after apply_manual_credentials().')
else:
    login(add_to_git_credential=False)
    print('Hugging Face login complete.')

if not os.environ.get('OPENROUTER_API_KEY'):
    raise RuntimeError(
        'OpenRouter credential is missing. Set OPENROUTER_API_KEY in the shell environment '
        'or paste it into MANUAL_OPENROUTER_API_KEY in section 2, then rerun sections 2 and 3.'
    )
print('OpenRouter judge credential is set.')

## 4. Configure one persistent run directory

In [ ]:
RUN_MODE = 'smoke'  # use 'full' only after smoke succeeds
MODEL_KEY = 'qwen35_9b'  # default local experiment model
CONFIG_NAME = f'phase1_{MODEL_KEY}_{RUN_MODE}'
RUN_NAME = f'jspace-{MODEL_KEY}-{RUN_MODE}'

RUN_ROOT = REPO_ROOT / 'artifacts' / RUN_NAME
PHASE1_DIR = RUN_ROOT / 'phase1'
PHASE2_DIR = RUN_ROOT / 'phase2'
PHASE3_DIR = RUN_ROOT / 'phase3'
PHASE4_DIR = RUN_ROOT / 'phase4'
BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'
CONFIG_PATH = REPO_ROOT / 'configs' / f'{CONFIG_NAME}.yaml'
WEBQA_TRAIN_PATH = None  # required for full mode
SUMMARIZATION_TRAIN_PATH = None  # required for full mode

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Configuration not found: {CONFIG_PATH}')


def run_command(command, label):
    if command and command[0].startswith('jspace-'):
        command = [str(VENV_BIN / command[0]), *command[1:]]
    print('Running:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with exit status {return_code}; see the traceback above.')


RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Config:', CONFIG_PATH)
print('Run root:', RUN_ROOT)

## 5. Verify the GPU runtime

In [ ]:
import torch

assert torch.cuda.is_available(), 'Phase 1 capture/analyze and Phase 2/4 generate require a CUDA GPU.'
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

## 6. Run or resume Phase 1

This freezes the manifest, captures activations, reconstructs J-space, and selects the layer. Rerunning the cell reuses compatible caches.

In [ ]:
phase1_command = [
    'jspace-phase1',
    '--config', str(CONFIG_PATH),
    '--output-dir', str(PHASE1_DIR),
    '--stage', 'all',
]
if WEBQA_TRAIN_PATH is not None:
    phase1_command.extend(['--webqa-train', str(WEBQA_TRAIN_PATH)])
if SUMMARIZATION_TRAIN_PATH is not None:
    phase1_command.extend(['--summarization-train', str(SUMMARIZATION_TRAIN_PATH)])
run_command(phase1_command, 'Phase 1')

## 7. Inspect Phase 1 before continuing

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

selection = json.loads((PHASE1_DIR / 'selected_layer.json').read_text())
print(json.dumps(selection, indent=2))
display(pd.read_csv(PHASE1_DIR / 'layer_metrics.csv'))
display(Image(filename=str(PHASE1_DIR / 'layer_auprc.png')))
display(Image(filename=str(PHASE1_DIR / 'selected_layer_score_distribution.png')))

## 8. Run or resume Phase 2 generation

This GPU stage reads the frozen Phase 1 directory directly and runs three conditions: intact (`alpha=0.0`), partial removal (`alpha=0.5`), and full removal (`alpha=1.0`).

In [ ]:
phase2_base = [
    'jspace-phase2',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE2_DIR),
]
run_command([*phase2_base, '--stage', 'generate'], 'Phase 2 generation')

## 9. Run or resume Phase 2 analysis

This stage uses cached generations, ROUGE scoring, and the pinned OpenRouter judge.

In [ ]:
run_command([*phase2_base, '--stage', 'analyze'], 'Phase 2 analysis')

## 10. Inspect Phase 2 results

In [ ]:
display(pd.read_csv(PHASE2_DIR / 'phase2_summary.csv'))
display(pd.read_csv(PHASE2_DIR / 'phase2_examples.csv'))
display(Image(filename=str(PHASE2_DIR / 'phase2_asr_vs_alpha.png')))
display(Image(filename=str(PHASE2_DIR / 'phase2_clean_utility_vs_alpha.png')))

## 11. Construct and inspect the Phase 3 detectors

This CPU-only stage reads the frozen Phase 1 handoff directly. It does not load Gemma or the lens and does not depend on Phase 2.

In [ ]:
phase3_command = [
    'jspace-phase3',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE3_DIR),
]
run_command(phase3_command, 'Phase 3')
display(pd.read_csv(PHASE3_DIR / 'phase3_metrics.csv'))
display(Image(filename=str(PHASE3_DIR / 'phase3_detector_comparison.png')))

## 12. Run or resume and inspect Phase 4

This runs the three frozen transfer benchmarks. Generation requires CUDA; analysis uses CPU and OpenRouter only for BIPIA semantic outcomes.

In [ ]:
phase4_base = [
    'jspace-phase4',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--phase3', str(PHASE3_DIR),
    '--bipia-root', str(BIPIA_ROOT),
    '--agentdojo-root', str(AGENTDOJO_CHECKOUT),
    '--injecagent-root', str(INJECAGENT_CHECKOUT),
    '--output-dir', str(PHASE4_DIR),
]
run_command([*phase4_base, '--stage', 'generate'], 'Phase 4 generation')
run_command([*phase4_base, '--stage', 'analyze'], 'Phase 4 analysis')
display(pd.read_csv(PHASE4_DIR / 'phase4_metrics.csv'))
display(Image(filename=str(PHASE4_DIR / 'phase4_detector_transfer.png')))

## 13. Confirm persistence

All caches and results are written under the local run root. Preserve the `phase1/`, `phase2/`, `phase3/`, and `phase4/` directories together when copying or archiving a run.

In [ ]:
print('Complete run root:', RUN_ROOT)
print('Phase 1 selected layer:', selection['selected_layer'])
print('Phase 2 results:', PHASE2_DIR / 'phase2_results.parquet')
print('Phase 3 metrics:', PHASE3_DIR / 'phase3_metrics.csv')
print('Phase 4 metrics:', PHASE4_DIR / 'phase4_metrics.csv')
assert (PHASE1_DIR / 'selected_layer.json').is_file()
assert (PHASE2_DIR / 'phase2_results.parquet').is_file()
assert (PHASE3_DIR / 'mean_detector.pt').is_file()
assert (PHASE3_DIR / 'logistic_detector.pt').is_file()
assert (PHASE4_DIR / 'phase4_predictions.parquet').is_file()

## Interpretation boundary

An effect in Phase 2 shows that the selected layer's reconstructed J-space component is functionally involved in model behavior. Phase 3 measures development-set detectability and freezes thresholds. Phase 4 evaluates those frozen detectors on held-out and transfer benchmarks without tuning. None of these phases establishes injection-specific causality.